# SIEVE / SPICE — Reproduction et extension de la classification de logs SIEM

**Projet de recherche — Master 2 IABD (ESP / UCAD)**
Article de référence : *Artioli et al., « SIEVE: Generating a cybersecurity log dataset collection for SIEM event classification », Computer Networks 266 (2025) 111330.*

## Objectifs
1. **Reproduire** le benchmark des auteurs (Logistic Regression, SVM, LSTM, BERT) sur les logs fournis.
2. **Étendre** l'étude : RNN, DistilBERT, comparaison de protocoles d'évaluation, test sur logs réels, robustesse au bruit.
3. **Évaluer rigoureusement** : métriques complètes, matrices de confusion, temps d'entraînement, intervalles de confiance.

## Démarche (ordre imposé, aucun entraînement avant l'EDA)
| Étape | Contenu |
|---|---|
| 0 | Configuration, reproductibilité, découverte **sans hypothèse** des fichiers |
| 1 | Chargement générique (détection automatique du format, vérifiable et surchargeable) |
| 2 | **EDA** : structure, classes, valeurs manquantes, doublons, distribution des labels, longueurs, nuage de mots |
| 3 | Prétraitement fidèle à l'article (IP/URL, dates, chemins, ponctuation, stopwords, stemming Snowball) |
| 4 | Encodage des labels et séparation Train/Test selon **3 protocoles** |
| 5 | Baseline (Dummy, Naive Bayes) |
| 6 | Modèles : Logistic Regression, SVM, RNN, LSTM, BERT Base, DistilBERT |
| 7 | Comparaison globale (F1, temps) |
| 8 | Extensions de recherche : entropie SPICE, fuite de données, robustesse au bruit |

## Les 3 protocoles d'évaluation
- **`paper_random`** : split aléatoire stratifié 75/25 sur les logs synthétiques (protocole de l'article).
- **`grouped`** : split 75/25 **par groupes de logs quasi identiques** (approximation du « template » d'origine). Il mesure la généralisation à de *nouveaux* gabarits et non à de simples variantes d'un gabarit déjà vu. C'est l'apport méthodologique principal de ce travail.
- **`real_test`** : entraînement sur le dataset synthétique, test sur les **logs réels** (le CDS de l'article), s'il est présent dans `data/`.

> **Mode d'emploi** : place les fichiers de logs dans `./data/` (à côté de ce notebook), puis exécute les cellules **dans l'ordre**. Si la détection automatique du format se trompe, ajuste uniquement la cellule `CFG` (§0.1). Rien d'autre n'est à modifier.

## 0. Configuration et reproductibilité

### 0.1 Paramètres globaux
Tous les réglages sont centralisés dans l'objet `CFG`. Les valeurs par défaut sont dimensionnées pour un portable avec **GPU 4 Go** (RTX 3050 Ti) : BERT/DistilBERT sont entraînés en précision mixte (fp16), avec séquences courtes, petit batch et accumulation de gradient.

- `QUICK_MODE=True` : exécution de contrôle en quelques minutes (sous-échantillons, 1-2 époques). **À utiliser pour vérifier que tout tourne**, puis repasser à `False`.
- Si le format n'est pas reconnu : renseigner `TEXT_COL` / `LABEL_COL` (fichiers tabulaires) ou `LABEL_FROM` / `LINE_SEPARATOR` (fichiers `.log`/`.txt`).

In [ ]:
from __future__ import annotations

import copy
import json
import os
import platform
import random
import re
import sys
import time
import warnings
from collections import Counter
from functools import lru_cache
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="notebook")

# ------------------------------------------------------------------ CONFIG
CFG = SimpleNamespace(
    # --- chemins (relatifs : le projet reste portable)
    DATA_DIR="./data",
    OUT_DIR="./outputs",
    SEED=42,
    QUICK_MODE=os.environ.get("SIEVE_QUICK", "0") == "1",   # mettre True pour un test rapide

    # --- lecture des données (None = détection automatique, à vérifier au §1)
    TEXT_COL=None,            # nom de la colonne contenant le log brut
    LABEL_COL=None,           # nom de la colonne contenant la classe
    HEADER=None,              # None=auto | True | False  (CSV/TSV)
    LABEL_FROM="auto",        # fichiers .log/.txt : "auto" | "filename" | "parent_folder"
    LINE_SEPARATOR=None,      # fichiers .log/.txt de type "label<SEP>log" (ex. "\t" ou ",")
    LINE_LABEL_POSITION="first",   # "first" ou "last" (position du label dans la ligne)
    REAL_DATASET_REGEX=r"cds|real|combined",   # nom du dataset de LOGS RÉELS (test externe)
    ACTIVE_DATASET=None,      # nom du dataset d'entraînement (None = le plus grand non réel)
    MAX_ROWS_PER_DATASET=None,

    # --- prétraitement
    DROP_DUPLICATES=True,     # supprime les doublons exacts (log, label)
    DROP_NUMBERS=False,       # l'article conserve les nombres (ex. "2808")

    # --- protocoles
    TEST_SIZE=0.25,           # 75/25 comme l'article
    PROTOCOLS=None,           # None = tous ceux disponibles

    # --- modèles classiques
    TUNE=False,               # True = petite GridSearch sur C (LR / LinearSVC)
    SVM_RBF_MAX_TRAIN=15000,  # SVC RBF (celui de l'article) : sous-échantillon car O(n²)

    # --- RNN / LSTM (PyTorch)
    RNN_MAX_TRAIN=None, RNN_EPOCHS=6, RNN_BATCH=256, RNN_LR=2e-3,
    EMB_DIM=50, HIDDEN=128, LAYERS=2, RNN_MAX_LEN=64,

    # --- Transformers (GPU 4 Go : batch 16 x accumulation 2 = batch effectif 32)
    BERT_CHECKPOINT="bert-base-uncased",
    DISTIL_CHECKPOINT="distilbert-base-uncased",
    TF_MAX_TRAIN=30000, TF_EPOCHS=2, TF_BATCH=16, TF_ACCUM=2, TF_LR=2e-5, TF_MAX_LEN_CAP=128,

    # --- extensions
    ENTROPY_MAX_TRAIN=30000,
)

if CFG.QUICK_MODE:                       # exécution de contrôle
    CFG.MAX_ROWS_PER_DATASET = 4000
    CFG.SVM_RBF_MAX_TRAIN = 1500
    CFG.RNN_EPOCHS = 2
    CFG.TF_MAX_TRAIN, CFG.TF_EPOCHS, CFG.TF_BATCH, CFG.TF_ACCUM = 600, 1, 8, 1
    CFG.ENTROPY_MAX_TRAIN = 3000

OUT = Path(CFG.OUT_DIR)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------- PyTorch (optionnel)
try:
    import torch
    HAS_TORCH = True
except Exception as exc:                 # ImportError ou DLL manquante (Windows)
    torch, HAS_TORCH = None, False
    print(f"[info] PyTorch indisponible ({type(exc).__name__}) : les sections RNN/LSTM/BERT/DistilBERT seront ignorées.")

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False


def set_seed(seed: int = CFG.SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    if HAS_TORCH and torch.cuda.is_available():
        return "cuda"
    if HAS_TORCH and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


set_seed()
DEVICE = get_device()

print(f"Python {sys.version.split()[0]} | {platform.system()} | seed={CFG.SEED} | QUICK_MODE={CFG.QUICK_MODE}")
print(f"pandas {pd.__version__} | numpy {np.__version__}")
if HAS_TORCH:
    print(f"torch {torch.__version__} | device = {DEVICE}")
    if DEVICE == "cuda":
        props = torch.cuda.get_device_properties(0)
        print(f"GPU : {props.name} | VRAM : {props.total_memory / 1e9:.1f} Go")
print(f"transformers disponible : {HAS_TRANSFORMERS}")

## 1. Découverte et chargement des données

### 1.1 Inventaire brut (aucune hypothèse sur la structure)
On liste d'abord **tous** les fichiers de `./data`, leur taille, leur nombre de lignes et quelques lignes brutes. Cette cellule ne fait que *regarder* : elle permet de vérifier que le format détecté ensuite est le bon.

In [ ]:
DATA_DIR = Path(CFG.DATA_DIR)
assert DATA_DIR.exists(), (
    f"Dossier introuvable : {DATA_DIR.resolve()}\n"
    "Crée un dossier 'data' à côté du notebook et place-y les fichiers de logs."
)

ALL_FILES = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and not p.name.startswith("."))
assert ALL_FILES, f"Aucun fichier dans {DATA_DIR.resolve()}"


def count_lines(path: Path, limit_mb: float = 300) -> int | None:
    if path.stat().st_size > limit_mb * 1e6:
        return None
    with path.open("rb") as f:
        return sum(1 for _ in f)


inventory = pd.DataFrame(
    {
        "fichier": [str(p.relative_to(DATA_DIR)) for p in ALL_FILES],
        "extension": [p.suffix.lower() or "(aucune)" for p in ALL_FILES],
        "taille_Mo": [round(p.stat().st_size / 1e6, 3) for p in ALL_FILES],
        "lignes": [count_lines(p) for p in ALL_FILES],
    }
)
display(inventory)

ARCHIVE_EXT = {".zip", ".gz", ".tar", ".tgz", ".7z", ".rar"}
archives = [p for p in ALL_FILES if p.suffix.lower() in ARCHIVE_EXT]
if archives:
    print("\n[ATTENTION] Archives détectées (non lues) : décompresse-les dans ./data :", [a.name for a in archives])

print("\n--- Aperçu brut (3 premières lignes des 12 premiers fichiers texte) ---")
for p in [p for p in ALL_FILES if p.suffix.lower() not in ARCHIVE_EXT | {".parquet", ".xlsx", ".xls"}][:12]:
    print(f"\n# {p.relative_to(DATA_DIR)}")
    with p.open("r", encoding="utf-8", errors="replace") as f:
        for _ in range(3):
            print("  ", repr(f.readline()[:220]))

### 1.2 Chargeur générique
Le chargeur reconnaît :
- **fichiers tabulaires** (`.csv`, `.tsv`, `.json`, `.jsonl`, `.parquet`, `.xlsx`) : une colonne texte + une colonne label, devinées par heuristique (longueur moyenne du texte, faible cardinalité du label, noms usuels). Un fichier = un dataset ;
- **fichiers de lignes** (`.log`, `.txt`) : le label est déduit du **dossier parent** ou du **nom du fichier** (ex. `authentication-failed.log`), ou lu dans la ligne si `LINE_SEPARATOR` est défini.

Le résultat est un dictionnaire `DATASETS[nom] -> DataFrame(text, label, source_file)`. Le tableau récapitulatif affiche les colonnes retenues : **vérifie-le** avant de continuer.

In [ ]:
TABULAR_EXT = {".csv", ".tsv", ".json", ".jsonl", ".parquet", ".xlsx", ".xls"}
LINE_EXT = {".log", ".txt", ".dat", ""}
TEXT_HINT = re.compile(r"log|text|message|msg|content|raw|event|line|entry", re.I)
LABEL_HINT = re.compile(r"label|class|categor|type|target|action|^y$", re.I)


def sniff_sep(path: Path, n: int = 4096) -> str:
    import csv
    with path.open("r", encoding="utf-8", errors="replace") as f:
        head = f.read(n)
    first = head.splitlines()[0] if head else ""
    for cand in [",", "\t", ";", "|"]:                  # en-tête à exactement 2 colonnes ?
        if cand in first and len(first.split(cand)) == 2:
            return cand
    try:
        return csv.Sniffer().sniff(head, delimiters=",;\t|").delimiter
    except csv.Error:
        return "\t" if head.count("\t") > head.count(",") else ","


def detect_header(path: Path, sep: str) -> bool:
    if CFG.HEADER is not None:
        return bool(CFG.HEADER)
    with path.open("r", encoding="utf-8", errors="replace") as f:
        first = f.readline().rstrip("\r\n")
    cells = [c.strip().strip('"') for c in first.split(sep)]
    short = [c for c in cells if len(c) < 40]
    return bool(short) and max(len(c) for c in cells) < 60 and any(
        TEXT_HINT.search(c) or LABEL_HINT.search(c) for c in short
    )


def read_table(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext in {".csv", ".tsv"}:
        sep = "\t" if ext == ".tsv" else sniff_sep(path)
        has_head = detect_header(path, sep)
        header = 0 if has_head else None
        # Cas fréquent pour des logs : fichier « label,log » où le log contient lui-même des virgules NON protégées.
        # Un parseur CSV standard perdrait ces lignes (ou les couperait en plusieurs colonnes) -> on découpe
        # chaque ligne au PREMIER séparateur seulement.
        if has_head:
            with path.open("r", encoding="utf-8", errors="replace") as f:
                head_cells = f.readline().rstrip("\r\n").split(sep)
            if len(head_cells) == 2:
                rows, no_sep = [], 0
                with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
                    f.readline()
                    for ln in f:
                        ln = ln.rstrip("\r\n")
                        if not ln.strip():
                            continue
                        i = ln.find(sep)
                        if i < 0:
                            no_sep += 1
                        else:
                            rows.append((ln[:i], ln[i + 1:]))
                print(f"[info] {path.name} : lu en mode 2 colonnes (découpage au premier « {sep!r} ») -> "
                      f"{len(rows)} lignes ; {no_sep} ligne(s) sans séparateur ignorée(s)")
                return pd.DataFrame(rows, columns=[c.strip().strip('"') for c in head_cells])
        try:
            df = pd.read_csv(path, sep=sep, header=header, dtype=str, encoding_errors="replace", on_bad_lines="warn")
        except Exception:
            df = pd.read_csv(path, sep=sep, header=header, dtype=str, encoding_errors="replace",
                             engine="python", on_bad_lines="skip")
        expected = (count_lines(path) or 0) - (1 if has_head else 0)
        if expected and len(df) < 0.999 * expected:
            print(f"[ATTENTION] {path.name} : {expected - len(df)} ligne(s) perdue(s) par le parseur CSV "
                  f"({len(df)} lues sur {expected}). Vérifie le format (virgules ou guillemets dans les logs).")
        if header is None:
            df.columns = [f"col_{i}" for i in range(df.shape[1])]
        return df
    if ext == ".jsonl":
        return pd.read_json(path, lines=True, dtype=False)
    if ext == ".json":
        try:
            return pd.read_json(path, dtype=False)
        except ValueError:
            return pd.read_json(path, lines=True, dtype=False)
    if ext == ".parquet":
        return pd.read_parquet(path)
    return pd.read_excel(path, dtype=str)


def guess_columns(df: pd.DataFrame) -> tuple[str, str | None]:
    cols = list(df.columns)
    mean_len = {c: df[c].astype(str).str.len().mean() for c in cols}
    nuniq = {c: df[c].nunique(dropna=True) for c in cols}
    text_col = CFG.TEXT_COL or max(cols, key=lambda c: mean_len[c] + (50 if TEXT_HINT.search(str(c)) else 0))
    label_col = CFG.LABEL_COL
    if label_col is None:
        cand = [c for c in cols if c != text_col and 2 <= nuniq[c] <= min(500, max(50, 0.05 * len(df))) and nuniq[c] < len(df)]
        if cand:
            label_col = sorted(cand, key=lambda c: (not LABEL_HINT.search(str(c)), nuniq[c]))[0]
    return text_col, label_col


def infer_label_mode(files: list[Path]) -> str:
    """Pour les fichiers .log/.txt : le label est-il le nom du fichier ou le dossier parent ?"""
    line_files = [p for p in files if p.suffix.lower() in LINE_EXT]
    rels = [p.relative_to(DATA_DIR).parts for p in line_files]
    if not rels or all(len(r) == 1 for r in rels):
        return "filename"
    if any(len(r) >= 3 for r in rels):
        return "parent_folder"                       # data/<dataset>/<classe>/<fichier>
    folders_of_stem: dict[str, set] = {}
    for r in rels:
        if len(r) == 2:
            folders_of_stem.setdefault(Path(r[1]).stem, set()).add(r[0])
    repeated = any(len(v) >= 2 for v in folders_of_stem.values())
    if not repeated:
        return "parent_folder"
    # mêmes noms de fichiers dans plusieurs dossiers : l'axe le plus riche (classes) porte le label,
    # l'autre (peu de valeurs) désigne les datasets
    n_folders = len({r[0] for r in rels if len(r) == 2})
    return "filename" if len(folders_of_stem) > n_folders else "parent_folder"


LINE_MODE = infer_label_mode(ALL_FILES)
print(f"Fichiers .log/.txt : label déduit du « {LINE_MODE} » (modifiable via CFG.LABEL_FROM)")


def read_lines_file(path: Path) -> pd.DataFrame:
    rel = path.relative_to(DATA_DIR)
    parts = rel.parts
    with path.open("r", encoding="utf-8", errors="replace") as f:
        lines = [ln.rstrip("\r\n") for ln in f if ln.strip()]
    if CFG.LINE_SEPARATOR:
        sep = CFG.LINE_SEPARATOR
        split = [ln.split(sep, 1) if CFG.LINE_LABEL_POSITION == "first" else ln.rsplit(sep, 1) for ln in lines]
        ok = [s for s in split if len(s) == 2]
        if CFG.LINE_LABEL_POSITION == "first":
            labels, texts = zip(*ok)
        else:
            texts, labels = zip(*ok)
        return pd.DataFrame({"text": texts, "label": labels, "source_file": str(rel)}), path.stem
    mode = CFG.LABEL_FROM if CFG.LABEL_FROM != "auto" else LINE_MODE
    if len(parts) == 1:
        mode = "filename"
    label = parts[-2] if mode == "parent_folder" and len(parts) >= 2 else path.stem
    if mode == "parent_folder":
        dataset = parts[-3] if len(parts) >= 3 else DATA_DIR.name
    else:
        dataset = parts[-2] if len(parts) >= 2 else DATA_DIR.name
    return pd.DataFrame({"text": lines, "label": label, "source_file": str(rel)}), dataset


def load_all() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    parts: dict[str, list[pd.DataFrame]] = {}
    summary = []
    for p in ALL_FILES:
        ext = p.suffix.lower()
        if ext in ARCHIVE_EXT:
            continue
        rel = str(p.relative_to(DATA_DIR))
        if ext in TABULAR_EXT:
            raw = read_table(p)
            if raw.shape[1] < 2 and not CFG.LINE_SEPARATOR:
                print(f"[skip] {rel} : une seule colonne ; indique TEXT_COL/LABEL_COL ou LINE_SEPARATOR.")
                continue
            tcol, lcol = guess_columns(raw)
            if lcol is None:
                print(f"[skip] {rel} : aucune colonne label plausible. Renseigne CFG.LABEL_COL (colonnes : {list(raw.columns)}).")
                continue
            part = pd.DataFrame({"text": raw[tcol], "label": raw[lcol], "source_file": rel})
            name, used = p.stem, f"text={tcol!r}, label={lcol!r}"
        elif ext in LINE_EXT:
            part, name = read_lines_file(p)
            used = f"label={'ligne' if CFG.LINE_SEPARATOR else CFG.LABEL_FROM}"
        else:
            print(f"[skip] {rel} : extension non gérée ({ext}).")
            continue
        parts.setdefault(name, []).append(part)
        summary.append({"fichier": rel, "dataset": name, "lignes": len(part), "colonnes_utilisees": used})
    datasets = {k: pd.concat(v, ignore_index=True) for k, v in parts.items()}
    return datasets, pd.DataFrame(summary)


def sample_per_class(df: pd.DataFrame, n_total: int, label_col: str = "label") -> pd.DataFrame:
    if n_total is None or len(df) <= n_total:
        return df
    return df.groupby(label_col, group_keys=False, sort=False).sample(frac=n_total / len(df), random_state=CFG.SEED)


DATASETS, load_summary = load_all()
assert DATASETS, "Aucun dataset chargé : voir les messages [skip] ci-dessus et ajuste CFG."
display(load_summary)

if CFG.MAX_ROWS_PER_DATASET:
    DATASETS = {k: sample_per_class(v, CFG.MAX_ROWS_PER_DATASET) for k, v in DATASETS.items()}

print("\nDatasets chargés :")
for k, v in DATASETS.items():
    print(f"  - {k:<25} {len(v):>8} logs | {v['label'].nunique():>3} classes")

### 1.3 Rôle de chaque dataset
- **Dataset réel (test externe)** : celui dont le nom correspond à `REAL_DATASET_REGEX` (par défaut `cds|real|combined`). C'est l'équivalent du CDS de l'article (440 logs réels).
- **Dataset actif (entraînement)** : le plus grand dataset restant, sauf si `CFG.ACTIVE_DATASET` est renseigné.

In [ ]:
real_candidates = [k for k in DATASETS if re.search(CFG.REAL_DATASET_REGEX, k, re.I)]
synthetic = [k for k in DATASETS if k not in real_candidates]

if CFG.ACTIVE_DATASET:
    ACTIVE_NAME = CFG.ACTIVE_DATASET
elif synthetic:
    ACTIVE_NAME = max(synthetic, key=lambda k: len(DATASETS[k]))
else:
    ACTIVE_NAME = max(DATASETS, key=lambda k: len(DATASETS[k]))
    real_candidates = []
REAL_NAME = next((k for k in real_candidates if k != ACTIVE_NAME), None)

print(f"Dataset ACTIF (train/test)  : {ACTIVE_NAME}")
print(f"Dataset RÉEL (test externe) : {REAL_NAME}")
print(f"Autres datasets (extension entropie) : {[k for k in DATASETS if k not in (ACTIVE_NAME, REAL_NAME)]}")

## 2. Analyse exploratoire (EDA)

Aucun modèle n'est entraîné avant la fin de cette section. La dernière cellule de l'EDA lève un *verrou* (`EDA_DONE`) : toutes les fonctions d'entraînement l'exigent.

### 2.1 Structure, classes, valeurs manquantes, doublons
On contrôle, **avant tout nettoyage** : dimensions, types, valeurs manquantes (texte et label), textes vides, doublons exacts, **conflits** (même log avec deux labels différents), nombre de classes et déséquilibre. Les labels sont normalisés (`strip`, minuscules, `_`/espace → `-`) pour pouvoir comparer le dataset synthétique et le dataset réel.

In [ ]:
def normalize_label(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.strip().str.lower()
              .str.replace(r"[\s_]+", "-", regex=True))


def audit(name: str, df: pd.DataFrame) -> dict:
    d = df.copy()
    d["label_n"] = normalize_label(d["label"].where(d["label"].notna(), "")).replace("", np.nan)
    text = d["text"].astype("string")
    stats = {
        "dataset": name,
        "lignes": len(d),
        "texte_manquant": int(text.isna().sum()),
        "texte_vide": int((text.fillna("").str.strip() == "").sum()),
        "label_manquant": int(d["label_n"].isna().sum()),
        "doublons_exacts(log,label)": int(d.duplicated(subset=["text", "label_n"]).sum()),
        "conflits(log->>1 label)": int((d.dropna(subset=["label_n"]).groupby("text")["label_n"].nunique() > 1).sum()),
        "n_classes": int(d["label_n"].nunique()),
    }
    vc = d["label_n"].value_counts()
    stats["ratio_desequilibre(max/min)"] = round(vc.max() / max(vc.min(), 1), 2) if len(vc) else np.nan
    return stats


audit_df = pd.DataFrame([audit(k, v) for k, v in DATASETS.items()]).set_index("dataset")
display(audit_df.T)

raw_active = DATASETS[ACTIVE_NAME]
print(f"\n=== Structure du dataset actif : {ACTIVE_NAME} ===")
raw_active.info()
display(raw_active.head(8))

In [ ]:
def clean_frame(raw: pd.DataFrame, drop_dups: bool = CFG.DROP_DUPLICATES) -> pd.DataFrame:
    d = raw[["text", "label", "source_file"]].copy()
    d["text"] = d["text"].astype("string").str.strip()
    d["label"] = normalize_label(d["label"].where(d["label"].notna(), ""))
    n0 = len(d)
    d = d[(d["text"].notna()) & (d["text"] != "") & (d["label"] != "") & (d["label"] != "nan")]
    n1 = len(d)
    if drop_dups:
        d = d.drop_duplicates(subset=["text", "label"])
    d["text"] = d["text"].astype(str)
    print(f"  nettoyage : {n0} -> {n1} (valeurs manquantes/vides) -> {len(d)} (doublons exacts)")
    return d.reset_index(drop=True)


print("Nettoyage minimal du dataset ACTIF :")
ACTIVE_RAW = clean_frame(raw_active)
REAL_RAW = None
if REAL_NAME:
    print("Nettoyage minimal du dataset RÉEL :")
    REAL_RAW = clean_frame(DATASETS[REAL_NAME], drop_dups=False)   # on garde tout le test réel

n_classes = ACTIVE_RAW["label"].nunique()
assert n_classes >= 2, (
    f"Une seule classe détectée ({ACTIVE_RAW['label'].unique()}). "
    "Vérifie CFG.LABEL_COL / CFG.LABEL_FROM / CFG.LINE_SEPARATOR."
)
print(f"\nClasses du dataset actif : {n_classes}")

if REAL_RAW is not None:
    only_real = sorted(set(REAL_RAW["label"]) - set(ACTIVE_RAW["label"]))
    only_syn = sorted(set(ACTIVE_RAW["label"]) - set(REAL_RAW["label"]))
    print(f"Classes présentes UNIQUEMENT dans le réel : {only_real}")
    print(f"Classes présentes UNIQUEMENT dans le synthétique : {only_syn}")
    if only_real:
        print("  -> si ces noms sont des variantes d'écriture, harmonise les labels avant de continuer.")

### 2.2 Distribution des classes
L'article annonce 30 classes **parfaitement équilibrées** pour SIEVE (3 334 ou 3 333 logs par classe) et un CDS réel très déséquilibré (de 2 à 93 logs par classe). Le graphique vérifie ce point sur *tes* données.

In [ ]:
def plot_class_distribution(df: pd.DataFrame, title: str, fname: str) -> None:
    vc = df["label"].value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.28 * len(vc))))
    ax.barh(vc.index, vc.values, color=sns.color_palette("viridis", len(vc)))
    for i, v in enumerate(vc.values):
        ax.text(v, i, f" {v}", va="center", fontsize=7)
    ax.set_title(title)
    ax.set_xlabel("Nombre de logs")
    plt.tight_layout()
    fig.savefig(OUT / "figures" / fname, dpi=150)
    plt.show()
    print(f"min={vc.min()} | max={vc.max()} | ratio max/min={vc.max() / vc.min():.2f}")


plot_class_distribution(ACTIVE_RAW, f"Distribution des classes — {ACTIVE_NAME}", "eda_class_distribution_active.png")
if REAL_RAW is not None:
    plot_class_distribution(REAL_RAW, f"Distribution des classes — {REAL_NAME} (réel)", "eda_class_distribution_real.png")

### 2.3 Longueur des logs
La longueur conditionne le choix de `max_length` pour les modèles séquentiels et les Transformers (compromis mémoire/GPU). On mesure caractères et tokens (séparés par espaces).

In [ ]:
def add_lengths(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["n_chars"] = d["text"].str.len()
    d["n_tokens"] = d["text"].str.split().str.len()
    return d


LEN_DF = add_lengths(ACTIVE_RAW)
display(LEN_DF[["n_chars", "n_tokens"]].describe(percentiles=[.5, .9, .95, .99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(LEN_DF["n_chars"], bins=60, ax=axes[0], color="#3b6ea5")
axes[0].set(title="Longueur en caractères", xlabel="caractères")
sns.histplot(LEN_DF["n_tokens"], bins=min(60, int(LEN_DF["n_tokens"].max())), ax=axes[1], color="#d9822b")
axes[1].set(title="Longueur en tokens (espaces)", xlabel="tokens")
plt.tight_layout()
fig.savefig(OUT / "figures" / "eda_log_length.png", dpi=150)
plt.show()

order = LEN_DF.groupby("label")["n_tokens"].median().sort_values().index
fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(order))))
sns.boxplot(data=LEN_DF, y="label", x="n_tokens", order=order, ax=ax, fliersize=1, color="#8fb8de")
ax.set(title="Longueur (tokens) par classe", xlabel="tokens", ylabel="")
plt.tight_layout()
fig.savefig(OUT / "figures" / "eda_log_length_by_class.png", dpi=150)
plt.show()

### 2.4 Vocabulaire et nuage de mots
Le nuage de mots est calculé après un nettoyage léger (lettres seules, longueur ≥ 3, stopwords usuels). On compare aussi le vocabulaire des logs **réels** à celui du dataset **synthétique** : le taux de mots réels *absents* de SIEVE (OOV) donne une première idée du risque de décalage de domaine, qui est la question centrale de l'expérience 3 de l'article.

In [ ]:
STOPWORDS = set(
    "a about above after again against all am an and any are as at be because been before being below between both "
    "but by can could did do does doing during each few for from further had has have having he her here hers him his "
    "how i if in into is it its itself just me more most my myself of on once only or other our over own same she "
    "should so some such than that the their theirs them then there these they this those through to too under until "
    "very was we were what when where which while who whom why will with would you your yours".split()
)   # volontairement SANS : no, not, nor, never, cannot, up, down, out, off (porteurs de sens en logs)

WORD_RE = re.compile(r"[A-Za-z]{3,}")


def light_tokens(text: str) -> list[str]:
    return [w for w in (m.lower() for m in WORD_RE.findall(text)) if w not in STOPWORDS]


all_tokens = Counter(w for t in ACTIVE_RAW["text"] for w in light_tokens(t))
print("Top 25 mots :", all_tokens.most_common(25))

try:
    from wordcloud import WordCloud
    wc = WordCloud(width=1100, height=450, background_color="white", max_words=150, random_state=CFG.SEED)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(wc.generate_from_frequencies(all_tokens), interpolation="bilinear")
    ax.axis("off"); ax.set_title(f"Nuage de mots — {ACTIVE_NAME}")
    fig.savefig(OUT / "figures" / "eda_wordcloud.png", dpi=150, bbox_inches="tight")
    plt.show()

    top_classes = ACTIVE_RAW["label"].value_counts().index[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    for ax, lab in zip(axes.ravel(), top_classes):
        freq = Counter(w for t in ACTIVE_RAW.loc[ACTIVE_RAW["label"] == lab, "text"] for w in light_tokens(t))
        if freq:
            ax.imshow(WordCloud(width=500, height=300, background_color="white", random_state=CFG.SEED)
                      .generate_from_frequencies(freq), interpolation="bilinear")
        ax.axis("off"); ax.set_title(lab, fontsize=10)
    plt.tight_layout()
    fig.savefig(OUT / "figures" / "eda_wordcloud_by_class.png", dpi=150)
    plt.show()
except ImportError:
    print("[info] 'wordcloud' non installé : affichage d'un diagramme des mots les plus fréquents à la place.")
    top = pd.Series(dict(all_tokens.most_common(30))).sort_values()
    top.plot.barh(figsize=(7, 7), title="Mots les plus fréquents")
    plt.show()

if REAL_RAW is not None:
    real_vocab = Counter(w for t in REAL_RAW["text"] for w in light_tokens(t))
    oov_types = [w for w in real_vocab if w not in all_tokens]
    oov_rate = sum(real_vocab[w] for w in oov_types) / max(1, sum(real_vocab.values()))
    print(f"\nVocabulaire RÉEL absent du synthétique : {len(oov_types)}/{len(real_vocab)} types "
          f"| {100 * oov_rate:.1f} % des occurrences")
    print("Exemples de mots réels absents :", oov_types[:20])

### 2.5 Verrou EDA
Cette cellule récapitule la liste de contrôle imposée avant tout entraînement. Si l'une des vérifications échoue, elle lève une erreur.

In [ ]:
EDA_DONE = {"flag": False}

checks = {
    "structure vérifiée (colonnes text/label)": {"text", "label"} <= set(ACTIVE_RAW.columns),
    "au moins 2 classes": ACTIVE_RAW["label"].nunique() >= 2,
    "aucune valeur manquante après nettoyage": not ACTIVE_RAW[["text", "label"]].isna().any().any(),
    "aucun texte vide": bool((ACTIVE_RAW["text"].str.strip() != "").all()),
    "distribution des labels analysée": True,
    "doublons exacts traités": (not CFG.DROP_DUPLICATES) or (not ACTIVE_RAW.duplicated(subset=["text", "label"]).any()),
}
for k, ok in checks.items():
    print(("[OK] " if ok else "[ÉCHEC] ") + k)
assert all(checks.values()), "EDA incomplète : corrige les points en échec avant d'entraîner."
EDA_DONE["flag"] = True


def require_eda() -> None:
    if not EDA_DONE["flag"]:
        raise RuntimeError("Exécute d'abord l'EDA (section 2) : aucun entraînement avant vérification des données.")


def torch_ready(need_transformers: bool = False) -> bool:
    if not HAS_TORCH:
        print("[skip] PyTorch indisponible dans cet environnement.")
        return False
    if need_transformers and not HAS_TRANSFORMERS:
        print("[skip] 'transformers' indisponible : pip install transformers")
        return False
    return True

### 2.6 Analyse des résultats de l'EDA (à rédiger après exécution)
Répondre dans cette cellule, comme dans la section « Dataset analysis » de l'article :
1. Les classes sont-elles équilibrées comme annoncé (SIEVE) ? Quel est le ratio max/min du dataset réel ?
2. Quelles longueurs (p95/p99) justifient le `max_length` retenu ? Les logs sont-ils plus longs que du texte naturel ?
3. Quelles classes partagent un vocabulaire proche (nuages de mots) ? Ce sont les candidates aux confusions.
4. Quel est le taux OOV réel → synthétique ? Un taux élevé annonce une chute de performance sur logs réels.
5. Combien de doublons et de conflits de labels ? Les conflits sont des erreurs d'annotation potentielles.

## 3. Prétraitement (fidèle à l'article, §6.1)

Pipeline appliqué **uniquement** aux modèles classiques (LR, SVM, NB) et aux modèles récurrents (RNN, LSTM). Les Transformers (BERT, DistilBERT) reçoivent le **log brut** : ils disposent de leur propre tokenisation, comme le précisent les auteurs.

| Étape | Rôle | Remarque |
|---|---|---|
| 1. Tokenisation | découpage en mots | espaces |
| 2. Suppression IP / URL | éléments variables non discriminants | regex |
| 3. Suppression dates / heures | idem | regex (l'article utilise `datefinder`) |
| 4. Suppression chemins et ponctuation | bruit | regex |
| 5. Stopwords | mots grammaticaux | liste ci-dessus, **négations conservées** |
| 6. Stemming Snowball | regroupe `install`/`installation` | NLTK `SnowballStemmer` |
| 7. TF-IDF + normalisation L2 | vectorisation | `TfidfVectorizer` ajusté **sur le train uniquement** |

> **Choix méthodologiques assumés** : (i) les mots `no`, `not`, `never`, `cannot`, `up`, `down`, `out`, `off` sont conservés car ils portent le sens (`authentication failed` vs `success`, `link down`). (ii) Le TF-IDF est appris sur le train seul pour éviter toute fuite d'information vers le test.

In [ ]:
from nltk.stem.snowball import SnowballStemmer

_MONTH = (r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|"
          r"sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)")
RE_URL = re.compile(r"(?:https?|ftp)://\S+|www\.\S+", re.I)
RE_IPV4 = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?\b")
RE_IPV6 = re.compile(r"\b(?:[0-9a-f]{1,4}:){2,7}[0-9a-f]{1,4}\b", re.I)
RE_DATE = re.compile(rf"\b{_MONTH}\s+\d{{1,2}}(?:,?\s+\d{{4}})?\b|\b\d{{4}}-\d{{2}}-\d{{2}}\b|\b\d{{1,2}}/\d{{1,2}}/\d{{2,4}}\b", re.I)
RE_TIME = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d+)?\b")
RE_PATH = re.compile(r"(?:[A-Za-z]:\\|~?/)[^\s:;,'\"]+")
RE_PUNCT = re.compile(r"[^\w\s]|_")

STEMMER = SnowballStemmer("english")


@lru_cache(maxsize=200_000)
def stem(word: str) -> str:
    return STEMMER.stem(word)


def preprocess(text: str) -> str:
    """Pipeline de nettoyage de l'article (IP/URL -> dates/heures -> chemins/ponctuation -> stopwords -> stemming)."""
    t = RE_URL.sub(" ", text)
    t = RE_IPV4.sub(" ", t)
    t = RE_IPV6.sub(" ", t)
    t = RE_DATE.sub(" ", t)
    t = RE_TIME.sub(" ", t)
    t = RE_PATH.sub(" ", t)
    t = RE_PUNCT.sub(" ", t).lower()
    tokens = [w for w in t.split() if w not in STOPWORDS]
    if CFG.DROP_NUMBERS:
        tokens = [w for w in tokens if not w.isdigit()]
    tokens = [stem(w) for w in tokens]
    return " ".join(tokens) if tokens else "emptylog"


# Démonstration étape par étape (comme le Tableau 3 de l'article) sur 3 logs du dataset actif
demo = ACTIVE_RAW["text"].sample(3, random_state=CFG.SEED).tolist()
for s in demo:
    print("BRUT   :", s)
    print("NETTOYÉ:", preprocess(s), "\n")

## 4. Encodage des labels et séparation Train/Test

### 4.1 Encodage et préparation
- Les labels sont encodés par un unique `LabelEncoder` ajusté sur l'**union** des classes (synthétique + réel), pour que les indices soient cohérents entre protocoles.
- Colonne `clean` : texte prétraité ; colonne `group` : clé de groupe utilisée par le protocole `grouped`.

### 4.2 La clé de groupe (approximation du « template »)
SIEVE est fabriqué à partir de **440 gabarits** réels : les ~100 000 logs sont des variantes de ces gabarits (valeurs Faker + perturbations EDA). Un split aléatoire place donc des variantes d'un même gabarit à la fois en train et en test (**fuite de gabarit**), ce qui gonfle les scores.
Faute de colonne `template_id`, on approche le gabarit ainsi : au sein de chaque classe, on ne garde que les tokens **alphabétiques fréquents** (présents dans ≥ 1 % des logs de la classe : les valeurs aléatoires Faker — noms d'utilisateur, hôtes, nombres — sont rares et écartées) puis on trie l'ensemble de ces tokens. Deux variantes qui diffèrent seulement par l'ordre (Random Swap) ou les valeurs aléatoires obtiennent la même clé. L'approximation est imparfaite et **sensible au seuil** `min_df_ratio` : sur SIEVE_00, des seuils différents donnent des groupes très différents (de ≈ 4 000 à ≈ 8 000 groupes, le plus gros regroupant jusqu'à ≈ 2 000 logs), et un très gros groupe placé en test retire presque toute une sous-famille de l'entraînement. Le protocole `grouped` donne donc un **ordre de grandeur** de la dépendance aux gabarits, **pas une mesure exacte** : consulter systématiquement le tableau ci-dessous (classes présentes en test, taille du plus gros groupe). Le protocole *open-set* par classes entières (phase suivante) n'a pas ce défaut.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

ALL_LABELS = set(ACTIVE_RAW["label"])
if REAL_RAW is not None:
    ALL_LABELS |= set(REAL_RAW["label"])
for k, v in DATASETS.items():                       # labels des autres datasets (extension entropie)
    ALL_LABELS |= set(normalize_label(v["label"].where(v["label"].notna(), "")))
ALL_LABELS -= {"", "nan"}
le = LabelEncoder().fit(sorted(ALL_LABELS))
N_CLASSES = len(le.classes_)
print(f"{N_CLASSES} classes encodées :", list(le.classes_))


def make_group_keys(df: pd.DataFrame, min_df_ratio: float = 0.01) -> pd.Series:
    """Clé de groupe = (classe, ensemble trié des tokens alphabétiques fréquents DANS la classe)."""
    keys = pd.Series("", index=df.index, dtype=object)
    for lab, sub in df.groupby("label"):
        cv = CountVectorizer(binary=True, token_pattern=r"[a-z]{2,}", lowercase=False,
                             min_df=max(3, int(min_df_ratio * len(sub))))
        try:
            vocab = set(cv.fit(sub["clean"]).vocabulary_)
        except ValueError:
            vocab = set()
        keys.loc[sub.index] = [lab + "|" + (" ".join(sorted({w for w in t.split() if w in vocab})) or t)
                               for t in sub["clean"]]
    return keys


def prepare_frame(raw: pd.DataFrame, with_groups: bool = False) -> pd.DataFrame:
    d = raw.copy()
    d["clean"] = d["text"].map(preprocess)
    d["y"] = le.transform(d["label"])
    if with_groups:
        d["group"] = pd.factorize(make_group_keys(d))[0]
    return d.reset_index(drop=True)


t0 = time.perf_counter()
ACTIVE = prepare_frame(ACTIVE_RAW, with_groups=True)
REAL = prepare_frame(REAL_RAW) if REAL_RAW is not None else None
print(f"Prétraitement terminé en {time.perf_counter() - t0:.1f} s")
print(f"Groupes (≈ gabarits) estimés : {ACTIVE['group'].nunique()} | taille médiane : "
      f"{int(ACTIVE.groupby('group').size().median())} logs/groupe")
display(ACTIVE[["text", "clean", "label", "y", "group"]].head(5))
print("Logs vides après nettoyage :", int((ACTIVE["clean"] == "emptylog").sum()))

In [ ]:
def _split_random(df: pd.DataFrame):
    try:
        tr, te = train_test_split(df, test_size=CFG.TEST_SIZE, stratify=df["y"], random_state=CFG.SEED)
    except ValueError:
        tr, te = train_test_split(df, test_size=CFG.TEST_SIZE, random_state=CFG.SEED)
    return tr.reset_index(drop=True), te.reset_index(drop=True)


def _split_grouped(df: pd.DataFrame):
    sgkf = StratifiedGroupKFold(n_splits=int(round(1 / CFG.TEST_SIZE)), shuffle=True, random_state=CFG.SEED)
    tr_idx, te_idx = next(sgkf.split(df, df["y"], df["group"]))
    return df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)


SPLITS: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {
    "paper_random": _split_random(ACTIVE),
    "grouped": _split_grouped(ACTIVE),
}
if REAL is not None:
    SPLITS["real_test"] = (ACTIVE, REAL)

PROTOCOLS = CFG.PROTOCOLS or list(SPLITS)
rows = []
for name, (tr, te) in SPLITS.items():
    rows.append({
        "protocole": name, "train": len(tr), "test": len(te),
        "classes_train": tr["y"].nunique(), "classes_test": te["y"].nunique(),
        "classes_test_absentes_du_train": len(set(te["y"]) - set(tr["y"])),
        "fuite_gabarit(%)": round(100 * te["group"].isin(set(tr["group"])).mean(), 1) if "group" in te else None,
    })
display(pd.DataFrame(rows).set_index("protocole"))
print("Protocoles exécutés :", PROTOCOLS)

# Diagnostics du protocole grouped (il est sensible à la clé de groupe : voir texte ci-dessus)
sizes_g = ACTIVE.groupby("group").size()
print(f"Groupes : {len(sizes_g)} | plus gros groupe : {int(sizes_g.max())} logs ({100 * sizes_g.max() / len(ACTIVE):.1f} % du jeu)")
tr_g, te_g = SPLITS["grouped"]
if te_g["y"].nunique() < ACTIVE["y"].nunique():
    print(f"[ATTENTION] grouped : {ACTIVE['y'].nunique() - te_g['y'].nunique()} classe(s) absente(s) du test ; interpréter les scores avec prudence.")

**Lecture du tableau** : la colonne `fuite_gabarit(%)` donne la part des logs de test dont le groupe figure aussi dans le train. Elle doit être très élevée pour `paper_random` et faible pour `grouped`. C'est la mesure directe de l'optimisme du protocole de l'article.

## 5. Utilitaires d'évaluation et de visualisation

Métriques calculées pour **chaque** modèle et chaque protocole : Accuracy, Precision, Recall, F1 (macro, micro et pondéré), temps d'entraînement et d'inférence, matrice de confusion.

> **Rappel important** : pour une classification mono-label multiclasse, `micro-F1 = micro-précision = micro-rappel = accuracy`. L'article rapporte ce micro-F1 (noté MaF1). On ajoute le **F1 macro**, plus exigeant car il pèse chaque classe de la même façon, ce qui compte pour le CDS très déséquilibré.

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_recall_fscore_support, precision_score, recall_score)

RESULTS: list[dict] = []
PREDS: dict[tuple[str, str], tuple[np.ndarray, np.ndarray]] = {}
MODELS: dict[tuple[str, str], object] = {}


def bootstrap_ci(y_true, y_pred, n_boot: int = 500, seed: int = CFG.SEED):
    """IC 95 % bootstrap du F1 macro (utile sur le petit jeu réel de ~440 logs)."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        vals.append(f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def record(model: str, protocol: str, y_true, y_pred, t_train: float, t_infer: float, notes: str = "") -> dict:
    row = {
        "model": model, "protocol": protocol, "dataset": ACTIVE_NAME,
        "n_train": len(SPLITS[protocol][0]), "n_test": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "train_time_s": t_train, "infer_time_s": t_infer, "notes": notes,
    }
    if protocol == "real_test":
        row["f1_macro_ci95"] = bootstrap_ci(y_true, y_pred)
    RESULTS.append(row)
    PREDS[(model, protocol)] = (np.asarray(y_true), np.asarray(y_pred))
    return row


def results_df() -> pd.DataFrame:
    return pd.DataFrame(RESULTS)


def top_confusions(y_true, y_pred, k: int = 8) -> pd.DataFrame:
    c = Counter((le.classes_[a], le.classes_[b]) for a, b in zip(y_true, y_pred) if a != b)
    return pd.DataFrame([{"vrai": a, "prédit": b, "erreurs": n} for (a, b), n in c.most_common(k)])


def plot_confusion(model: str, protocol: str, normalize: bool = True) -> None:
    y_true, y_pred = PREDS[(model, protocol)]
    idx = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=idx).astype(float)
    shown = cm.copy()
    if normalize:
        rows_sum = cm.sum(axis=1, keepdims=True)
        shown = np.divide(cm, rows_sum, out=np.zeros_like(cm), where=rows_sum > 0)
    annot = np.where(shown > 0, np.round(shown, 2).astype(str) if normalize else shown.astype(int).astype(str), "")
    names = [le.classes_[i] for i in idx]
    size = max(7, 0.42 * len(idx))
    fig, ax = plt.subplots(figsize=(size + 2, size))
    sns.heatmap(shown, annot=annot, fmt="", cmap="Blues", xticklabels=names, yticklabels=names,
                cbar=True, ax=ax, annot_kws={"size": 6}, linewidths=0.2, linecolor="#eeeeee")
    ax.set(xlabel="Classe prédite", ylabel="Classe réelle",
           title=f"Matrice de confusion {'normalisée' if normalize else ''} — {model} [{protocol}]")
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout()
    fig.savefig(OUT / "figures" / f"cm_{model}_{protocol}.png".replace(" ", "_"), dpi=150)
    plt.show()


def show_results(model: str, protocol: str) -> None:
    df = results_df()
    r = df[(df["model"] == model) & (df["protocol"] == protocol)].iloc[-1]
    cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_micro", "f1_weighted",
            "train_time_s", "infer_time_s"]
    print(f"\n==== {model} | protocole = {protocol} | train={r['n_train']} | test={r['n_test']} ====")
    display(r[cols].to_frame("valeur").T.round(4))
    if protocol == "real_test":
        lo, hi = r["f1_macro_ci95"]
        print(f"IC 95 % bootstrap du F1 macro : [{lo:.3f} ; {hi:.3f}]  (n_test = {r['n_test']})")
    y_true, y_pred = PREDS[(model, protocol)]
    present = sorted(set(y_true) | set(y_pred))
    print(classification_report(y_true, y_pred, labels=present, target_names=[le.classes_[i] for i in present],
                                zero_division=0, digits=3))
    plot_confusion(model, protocol)
    print("Principales confusions :")
    display(top_confusions(y_true, y_pred))


KNOWN_CONFUSIONS = [("directory-deleted", "file-deleted"), ("directory-changed", "file-modification"),
                    ("user-logout", "connection-closed"), ("authentication-failed", "authentication-success")]
PAPER_REF = {   # Tableaux 7 et 10 de l'article (SIEVE_00, 100 000 logs, micro-F1)
    "LogisticRegression": {"paper_random": 0.9720, "real_test": 0.9681},
    "LinearSVC": {"paper_random": 0.9737, "real_test": 0.9636},
    "SVM-RBF": {"paper_random": 0.9737, "real_test": 0.9636},
    "LSTM": {"paper_random": 0.9715, "real_test": 0.9524},
    "BERT": {"paper_random": 0.9730, "real_test": 0.9182},
    "NaiveBayes": {"paper_random": 0.9335, "real_test": 0.8909},
}


def interpret(model: str, protocol: str) -> None:
    """Interprétation automatique factuelle (à compléter par l'analyse rédigée)."""
    df = results_df()
    r = df[(df["model"] == model) & (df["protocol"] == protocol)].iloc[-1]
    y_true, y_pred = PREDS[(model, protocol)]
    present = sorted(set(y_true))
    _, _, f1c, _ = precision_recall_fscore_support(y_true, y_pred, labels=present, zero_division=0)
    worst = sorted(zip(f1c, present))[:3]
    print(f"• {model} [{protocol}] : F1 macro = {r['f1_macro']:.4f} | accuracy (= micro-F1) = {r['accuracy']:.4f}")
    print("• Classes les plus difficiles (F1) : " + ", ".join(f"{le.classes_[c]} ({f:.2f})" for f, c in worst))
    tc = top_confusions(y_true, y_pred, 3)
    if len(tc):
        print("• Confusions dominantes : " + "; ".join(f"{a}->{b} (x{n})" for a, b, n in tc.values))
    names = set(le.classes_)
    seen = [(a, b) for a, b in KNOWN_CONFUSIONS
            if a in names and b in names and any({le.classes_[t], le.classes_[p]} == {a, b}
                                                 for t, p in zip(y_true, y_pred) if t != p)]
    if seen:
        print("• Confusions signalées par les auteurs également observées ici : " + ", ".join(f"{a}<->{b}" for a, b in seen))
    ref = PAPER_REF.get(model, {}).get(protocol)
    if ref:
        print(f"• Référence article (micro-F1) : {ref:.4f} -> écart = {r['accuracy'] - ref:+.4f}")
    if protocol == "paper_random" and ((df["model"] == model) & (df["protocol"] == "grouped")).any():
        g = df[(df["model"] == model) & (df["protocol"] == "grouped")].iloc[-1]
        print(f"• Optimisme du split aléatoire : F1 macro {r['f1_macro']:.4f} (random) vs {g['f1_macro']:.4f} (grouped) "
              f"-> écart = {r['f1_macro'] - g['f1_macro']:+.4f}")

## 6. Modèles classiques

Fonction commune : un `Pipeline` (TF-IDF → classifieur) ajusté **sur le train uniquement**, chronométré, évalué sur le test du protocole.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC


def strat_subsample(df: pd.DataFrame, n: int | None) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df
    return df.groupby("y", group_keys=False, sort=False).sample(frac=n / len(df), random_state=CFG.SEED)


def make_tfidf() -> TfidfVectorizer:
    # tokens déjà nettoyés/stemmés : on découpe sur les espaces ; TF-IDF + norme L2 comme l'article (Eq. 3-5)
    return TfidfVectorizer(lowercase=False, token_pattern=r"\S+", norm="l2", smooth_idf=True)


def run_sklearn(name: str, make_estimator, protocol: str, max_train: int | None = None, notes: str = "") -> Pipeline:
    require_eda()
    set_seed()
    tr, te = SPLITS[protocol]
    tr = strat_subsample(tr, max_train)
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", make_estimator())])
    t0 = time.perf_counter()
    pipe.fit(tr["clean"], tr["y"])
    t_train = time.perf_counter() - t0
    t0 = time.perf_counter()
    pred = pipe.predict(te["clean"])
    t_infer = time.perf_counter() - t0
    record(name, protocol, te["y"].to_numpy(), pred, t_train, t_infer,
           notes or (f"train={len(tr)}" if max_train else ""))
    MODELS[(name, protocol)] = pipe
    print(f"[{name:<20}|{protocol:<12}] F1 macro={RESULTS[-1]['f1_macro']:.4f} | acc={RESULTS[-1]['accuracy']:.4f} "
          f"| train={t_train:.1f}s | infer={t_infer:.2f}s")
    return pipe


def tune_C(make_clf, protocol: str, grid=(0.1, 1.0, 10.0), max_rows: int = 20000) -> float:
    """Mini GridSearch (3-fold, F1 macro) sur sous-échantillon : l'article choisit ses paramètres par GridSearch."""
    tr = strat_subsample(SPLITS[protocol][0], max_rows)
    gs = GridSearchCV(Pipeline([("tfidf", make_tfidf()), ("clf", make_clf())]),
                      {"clf__C": list(grid)}, scoring="f1_macro", cv=3, n_jobs=1)
    gs.fit(tr["clean"], tr["y"])
    print(f"  GridSearch C -> meilleur C = {gs.best_params_['clf__C']} (F1 macro CV = {gs.best_score_:.4f})")
    return gs.best_params_["clf__C"]

### 6.0 Baseline simple : Dummy + Naive Bayes

**Pourquoi** : toute étude sérieuse commence par une borne basse. Le classifieur `DummyClassifier(most_frequent)` donne le plancher absolu (hasard informé). **Naive Bayes** multinomial est la baseline textuelle classique, et c'est aussi le modèle le plus rapide et le plus faible de l'article (MaF1 0,9335 sur SIEVE_00 ; 0,8909 sur logs réels).

- **Principe** : $P(c\mid d)\propto P(c)\prod_{t}P(t\mid c)^{x_t}$ avec lissage de Laplace ($\alpha=1$) ; indépendance conditionnelle des mots.
- **Avantages** : entraînement quasi instantané, aucun réglage, bonne base de comparaison.
- **Limites** : hypothèse d'indépendance fausse (l'ordre `login failed` vs `failed login` est perdu), mauvaise calibration, sensible aux mots très fréquents.
- **Complexité** : entraînement $O(n\cdot \bar{L})$ (un passage), prédiction $O(k\cdot \bar{L})$.

In [ ]:
for proto in PROTOCOLS:
    run_sklearn("Dummy", lambda: DummyClassifier(strategy="most_frequent"), proto)
    run_sklearn("NaiveBayes", lambda: MultinomialNB(alpha=1.0), proto)

In [ ]:
for proto in PROTOCOLS:
    show_results("NaiveBayes", proto)
    interpret("NaiveBayes", proto)

**Analyse (à rédiger)** : comparer le plancher `Dummy` au NB. Un NB déjà très au-dessus du hasard signifie que le problème est en grande partie lexical ; un écart important entre `paper_random` et `grouped` signale une dépendance aux gabarits mémorisés. Vérifier que NB reste le plus faible, comme dans l'article.

**Étape suivante** : Logistic Regression, le modèle linéaire discriminant de référence.

### 6.1 Logistic Regression

**Principe théorique.** Modèle linéaire discriminant : pour $k$ classes, la régression logistique *multinomiale* modélise $P(c\mid x)=\mathrm{softmax}(Wx+b)_c$ et minimise l'entropie croisée régularisée $\mathcal{L}=-\sum_i \log P(y_i\mid x_i)+\frac{1}{2C}\lVert W\rVert_2^2$. Optimiseur `lbfgs` (quasi-Newton), $C=1$, `max_iter=300` comme au Tableau 5 de l'article.

**Pourquoi elle est pertinente ici.** Les logs sont des textes courts à vocabulaire technique : un vecteur TF-IDF creux et un modèle linéaire suffisent souvent, car les mots-clés (`failed`, `deleted`, `session`) sont très discriminants. Dans l'article, LR est 2ᵉ/3ᵉ sur SIEVE (0,9720 sur SIEVE_00) et **meilleure sur logs réels** (0,9681 à $p=0$).

**Avantages.** Probabilités calibrées par classe, coefficients interprétables (mots les plus pondérés par classe), régularisation $\ell_2$ contre le surapprentissage, robuste au bruit synthétique.

**Limites.** Frontières linéaires seulement (pas d'interactions entre mots), ignore l'ordre des tokens ; entraînement plus lent que SGD (≈ 2 min pour 75 000 logs dans l'article).

**Complexité.** Entraînement $O(I\cdot \mathrm{nnz}\cdot k)$ ($I$ itérations lbfgs, $\mathrm{nnz}$ valeurs non nulles du TF-IDF, $k$ classes) ; prédiction $O(\mathrm{nnz}\cdot k)$ par document ; mémoire $O(d\cdot k)$.

**Intérêt par rapport aux autres modèles de l'article.** C'est la meilleure référence « simple et robuste » : presque aussi bonne que SVM, bien plus rapide, et plus généralisante que BERT sur logs réels.

In [ ]:
for proto in PROTOCOLS:
    C_lr = tune_C(lambda: LogisticRegression(max_iter=300, random_state=CFG.SEED), proto) if CFG.TUNE else 1.0
    run_sklearn("LogisticRegression",
                lambda: LogisticRegression(C=C_lr, max_iter=300, solver="lbfgs", random_state=CFG.SEED),
                proto, notes=f"C={C_lr}")

In [ ]:
for proto in PROTOCOLS:
    show_results("LogisticRegression", proto)
    interpret("LogisticRegression", proto)

# Interprétabilité : mots les plus pondérés par classe (protocole paper_random ou premier disponible)
proto0 = PROTOCOLS[0]
pipe = MODELS[("LogisticRegression", proto0)]
vocab = np.array(pipe["tfidf"].get_feature_names_out())
coef = pipe["clf"].coef_
top_words = {le.classes_[c]: ", ".join(vocab[np.argsort(coef[i])[::-1][:6]])
             for i, c in enumerate(pipe["clf"].classes_)}
display(pd.Series(top_words, name="mots les plus discriminants (stemmés)").to_frame().head(30))

**Analyse (à rédiger)** :
- Comparer le F1 macro / l'accuracy avec la référence de l'article (affichée par `interpret`) ; commenter l'écart.
- Interpréter l'écart `paper_random` → `grouped` : une chute marquée prouve que le score de l'article est en partie dû à la mémorisation des gabarits.
- Lire le tableau des mots discriminants : sont-ils sémantiquement cohérents (ex. `fail` pour `authentication-failed`) ou ce sont des artefacts de génération ?
- Dans `real_test`, relever l'intervalle de confiance : avec ~440 logs, une différence de 1 point entre deux modèles n'est généralement pas significative.

**Étape suivante** : SVM, meilleur modèle de l'article.

### 6.2 Support Vector Machine (SVM)

**Principe théorique.** Un SVM cherche l'hyperplan à **marge maximale** : $\min_{w,b}\frac12\lVert w\rVert^2 + C\sum_i \xi_i$ sous $y_i(w\cdot\phi(x_i)+b)\ge 1-\xi_i$. Avec un noyau RBF $K(x,x')=\exp(-\gamma\lVert x-x'\rVert^2)$, il sépare des classes non linéairement séparables. Le multiclasse est géré en *one-vs-rest* (`decision_function_shape="ovr"`).

Deux variantes sont étudiées :
- **`SVM-RBF`** : le `SVC(kernel="rbf", C=1, gamma="scale")` des auteurs (Tableau 5). Très coûteux ($O(n^2)$ à $O(n^3)$) : entraîné sur un **sous-échantillon** (`CFG.SVM_RBF_MAX_TRAIN`), ce qui est une limite à mentionner dans le mémoire.
- **`LinearSVC`** : SVM à noyau linéaire (liblinear), exploitable sur tout le jeu et adapté aux données TF-IDF creuses de grande dimension, où le noyau linéaire est souvent aussi bon que le RBF.

**Pourquoi il est pertinent ici.** Dans l'article, SVM est le **meilleur modèle global** : 0,9737 sur SIEVE_00 et 0,9636 sur logs réels, avec une dégradation très faible. Les textes TF-IDF sont de très grande dimension (≈ 120-165 k variables) avec peu de relations non linéaires : terrain idéal pour les SVM.

**Avantages.** Excellente généralisation grâce à la marge, efficace en grande dimension, peu de surapprentissage, robuste au bruit synthétique.

**Limites.** SVC RBF : entraînement très long (jusqu'à 18 min pour 75 000 logs dans l'article, contre 3 s pour SGD), mémoire $O(n^2)$ pour le cache de noyau, pas de probabilités natives, réglage de $C$ et $\gamma$.

**Complexité.** `SVC` : entraînement $O(n^2 d)\!\to\!O(n^3)$, prédiction $O(n_{sv}\cdot d)$. `LinearSVC` : $\approx O(n\cdot \mathrm{nnz}\cdot k)$ par époque, prédiction $O(\mathrm{nnz}\cdot k)$.

**Intérêt par rapport aux autres modèles.** Référence de performance *et* de robustesse : l'un des tests clés est de vérifier s'il conserve cet avantage sous le protocole `grouped`, où l'article n'a pas testé.

In [ ]:
for proto in PROTOCOLS:
    C_svm = tune_C(lambda: LinearSVC(random_state=CFG.SEED, max_iter=5000), proto) if CFG.TUNE else 1.0
    run_sklearn("LinearSVC", lambda: LinearSVC(C=C_svm, max_iter=5000, random_state=CFG.SEED), proto, notes=f"C={C_svm}")
    run_sklearn("SVM-RBF", lambda: SVC(C=1.0, kernel="rbf", gamma="scale", decision_function_shape="ovr"),
                proto, max_train=CFG.SVM_RBF_MAX_TRAIN, notes=f"sous-échantillon {CFG.SVM_RBF_MAX_TRAIN}")

In [ ]:
for proto in PROTOCOLS:
    for m in ("LinearSVC", "SVM-RBF"):
        show_results(m, proto)
        interpret(m, proto)

**Analyse (à rédiger)** :
- `LinearSVC` vs `SVM-RBF` : le noyau RBF apporte-t-il un gain ? À quel coût en temps (voir `train_time_s`) ? Rappeler que le SVC RBF ne voit ici qu'un sous-échantillon.
- Comparer à LR : sur TF-IDF, les deux modèles linéaires sont souvent proches ; la différence éventuelle vient de la perte (hinge vs log).
- Examiner les confusions (`directory-deleted`/`file-deleted`, `authentication-failed`/`authentication-success`) : ce sont des paires quasi identiques lexicalement, ce que l'article attribue à la proximité conceptuelle. Dans un SIEM, confondre *échec* et *succès* d'authentification masque une attaque par force brute.

**Étape suivante** : modèles séquentiels (RNN, LSTM), qui exploitent l'ordre des tokens.

## 7. Modèles de Deep Learning (PyTorch)

### 7.0 Outils communs
Une boucle d'entraînement générique sert aux quatre modèles (RNN, LSTM, BERT, DistilBERT) pour garantir une comparaison équitable :
- optimiseur **AdamW**, *gradient clipping* à 1,0 ;
- **précision mixte fp16** et **accumulation de gradient** (indispensables pour BERT sur un GPU de 4 Go) ;
- un **jeu de validation** (10 % du train) pour sélectionner la meilleure époque (*early stopping* avec patience) : le test n'est **jamais** utilisé pour choisir le modèle ;
- chronométrage de l'entraînement et de l'inférence.

> Si PyTorch n'est pas disponible, ces sections affichent un message `[skip]` et le reste du notebook continue.

In [ ]:
if HAS_TORCH:
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    try:
        from tqdm.auto import tqdm
    except ImportError:                                     # tqdm facultatif
        def tqdm(x, **kw):
            return x

    def split_val(df: pd.DataFrame, val_size: float = 0.1):
        try:
            a, b = train_test_split(df, test_size=val_size, stratify=df["y"], random_state=CFG.SEED)
        except ValueError:
            a, b = train_test_split(df, test_size=val_size, random_state=CFG.SEED)
        return a.reset_index(drop=True), b.reset_index(drop=True)

    def predict_torch(model, loader, forward_fn, device, use_amp=False):
        model.eval()
        ys, ps = [], []
        with torch.inference_mode():
            for batch in loader:
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    logits = forward_fn(model, batch, device)
                ps.append(logits.argmax(-1).cpu().numpy())
                ys.append(batch["labels"].cpu().numpy())
        return np.concatenate(ys), np.concatenate(ps)

    def fit_torch(model, train_loader, val_loader, forward_fn, *, epochs, lr, device, weight_decay=0.0,
                  grad_accum=1, use_amp=False, warmup=False, patience=2, desc="model") -> pd.DataFrame:
        model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        total_steps = max(1, (len(train_loader) // grad_accum) * epochs)
        sched = None
        if warmup:
            from transformers import get_linear_schedule_with_warmup
            sched = get_linear_schedule_with_warmup(opt, int(0.06 * total_steps), total_steps)
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        history, best, best_state, bad = [], -1.0, None, 0
        for ep in range(1, epochs + 1):
            model.train()
            running, seen = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for i, batch in enumerate(tqdm(train_loader, desc=f"{desc} époque {ep}/{epochs}", leave=False)):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    logits = forward_fn(model, batch, device)
                    loss = nn.functional.cross_entropy(logits.float(), batch["labels"].to(device)) / grad_accum
                scaler.scale(loss).backward()
                if (i + 1) % grad_accum == 0 or (i + 1) == len(train_loader):
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
                    if sched is not None:
                        sched.step()
                n = len(batch["labels"])
                running += loss.item() * grad_accum * n
                seen += n
            y_val, p_val = predict_torch(model, val_loader, forward_fn, device, use_amp)
            f1 = f1_score(y_val, p_val, average="macro", zero_division=0)
            history.append({"epoch": ep, "train_loss": running / seen, "val_f1_macro": f1})
            print(f"  {desc} | époque {ep}/{epochs} | loss={running / seen:.4f} | val F1 macro={f1:.4f}")
            if f1 > best:
                best, bad = f1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    print("  early stopping")
                    break
        model.load_state_dict(best_state)
        return pd.DataFrame(history)

    def count_params(model) -> float:
        return sum(p.numel() for p in model.parameters()) / 1e6

    HISTORIES: dict[tuple[str, str], pd.DataFrame] = {}

### 7.1 Briques pour RNN / LSTM
Les deux modèles partagent **exactement** la même architecture (embedding de dimension 50 → 2 couches récurrentes de 128 unités → couche linéaire), comme le LSTM de l'article ; seule la cellule change. Ainsi, l'écart RNN/LSTM mesure la contribution des **portes** et de la cellule mémoire.

- Vocabulaire construit **sur le train** (fréquence minimale 2, jeton `<UNK>` pour le reste) sur le texte prétraité ;
- séquences tronquées au 99ᵉ percentile (plafonné par `RNN_MAX_LEN`) ;
- `pack_padded_sequence` : le padding n'influence pas l'état final.

In [ ]:
if HAS_TORCH:
    class Vocab:
        PAD, UNK = 0, 1

        def __init__(self, min_freq: int = 2):
            self.min_freq, self.stoi = min_freq, {"<pad>": 0, "<unk>": 1}

        def fit(self, texts):
            counts = Counter(w for t in texts for w in t.split())
            for w, c in counts.most_common():
                if c >= self.min_freq:
                    self.stoi[w] = len(self.stoi)
            return self

        def encode(self, text: str, max_len: int) -> list[int]:
            ids = [self.stoi.get(w, self.UNK) for w in text.split()][:max_len]
            return ids or [self.UNK]

        def __len__(self):
            return len(self.stoi)

    class IdsDataset(Dataset):
        def __init__(self, ids, labels):
            self.ids, self.labels = ids, labels

        def __len__(self):
            return len(self.ids)

        def __getitem__(self, i):
            return self.ids[i], int(self.labels[i])

    def collate_ids(batch):
        seqs, labels = zip(*batch)
        lengths = torch.tensor([len(s) for s in seqs])
        padded = torch.zeros(len(seqs), int(lengths.max()), dtype=torch.long)
        for i, s in enumerate(seqs):
            padded[i, : len(s)] = torch.tensor(s)
        return {"x": padded, "lengths": lengths, "labels": torch.tensor(labels, dtype=torch.long)}

    class RecurrentClassifier(nn.Module):
        def __init__(self, vocab_size, n_classes, cell="LSTM", emb_dim=50, hidden=128, layers=2,
                     dropout=0.3, bidirectional=False):
            super().__init__()
            self.cell, self.bidirectional = cell, bidirectional
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            kw = dict(input_size=emb_dim, hidden_size=hidden, num_layers=layers, batch_first=True,
                      dropout=dropout if layers > 1 else 0.0, bidirectional=bidirectional)
            self.rnn = {"RNN": lambda: nn.RNN(nonlinearity="tanh", **kw), "LSTM": lambda: nn.LSTM(**kw),
                        "GRU": lambda: nn.GRU(**kw)}[cell]()
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden * (2 if bidirectional else 1), n_classes)

        def forward(self, x, lengths):
            e = self.drop(self.emb(x))
            packed = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, h = self.rnn(packed)
            h = h[0] if self.cell == "LSTM" else h              # LSTM renvoie (h, c)
            feat = torch.cat([h[-2], h[-1]], dim=1) if self.bidirectional else h[-1]
            return self.fc(self.drop(feat))

    def rnn_forward(model, batch, device):
        return model(batch["x"].to(device), batch["lengths"])

    def run_recurrent(name: str, cell: str, protocol: str):
        require_eda()
        set_seed()
        tr, te = SPLITS[protocol]
        tr = strat_subsample(tr, CFG.RNN_MAX_TRAIN)
        tr_fit, tr_val = split_val(tr)
        vocab = Vocab(min_freq=2).fit(tr_fit["clean"])
        lens = tr_fit["clean"].str.split().str.len()
        max_len = int(np.clip(np.percentile(lens, 99), 8, CFG.RNN_MAX_LEN))

        def make_loader(df, shuffle):
            ds = IdsDataset([vocab.encode(t, max_len) for t in df["clean"]], df["y"].to_numpy())
            return DataLoader(ds, batch_size=CFG.RNN_BATCH, shuffle=shuffle, collate_fn=collate_ids)

        train_loader, val_loader, test_loader = make_loader(tr_fit, True), make_loader(tr_val, False), make_loader(te, False)
        model = RecurrentClassifier(len(vocab), N_CLASSES, cell=cell, emb_dim=CFG.EMB_DIM,
                                    hidden=CFG.HIDDEN, layers=CFG.LAYERS)
        print(f"[{name}|{protocol}] vocab={len(vocab)} | max_len={max_len} | params={count_params(model):.2f} M | device={DEVICE}")
        t0 = time.perf_counter()
        hist = fit_torch(model, train_loader, val_loader, rnn_forward, epochs=CFG.RNN_EPOCHS, lr=CFG.RNN_LR,
                         device=DEVICE, weight_decay=1e-5, desc=name)
        t_train = time.perf_counter() - t0
        t0 = time.perf_counter()
        y_true, y_pred = predict_torch(model, test_loader, rnn_forward, DEVICE)
        t_infer = time.perf_counter() - t0
        record(name, protocol, y_true, y_pred, t_train, t_infer,
               f"vocab={len(vocab)}, max_len={max_len}, epochs={len(hist)}, params={count_params(model):.2f}M")
        HISTORIES[(name, protocol)] = hist
        print(f"[{name:<8}|{protocol:<12}] F1 macro={RESULTS[-1]['f1_macro']:.4f} | acc={RESULTS[-1]['accuracy']:.4f} "
              f"| train={t_train:.0f}s | infer={t_infer:.1f}s")
        return model


def plot_history(name: str, protocol: str) -> None:
    h = HISTORIES.get((name, protocol))
    if h is None:
        return
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(h["epoch"], h["train_loss"], marker="o", label="loss (train)")
    ax2 = ax.twinx()
    ax2.plot(h["epoch"], h["val_f1_macro"], marker="s", color="tab:orange", label="F1 macro (val)")
    ax.set(xlabel="époque", ylabel="loss", title=f"Courbes d'apprentissage — {name} [{protocol}]")
    ax2.set_ylabel("F1 macro (validation)")
    fig.tight_layout()
    fig.savefig(OUT / "figures" / f"history_{name}_{protocol}.png".replace(" ", "_"), dpi=150)
    plt.show()

### 7.2 RNN (Elman, tanh)

**Principe théorique.** Un réseau récurrent lit la séquence token par token et met à jour un état caché $h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b)$ ; l'état final $h_T$ résume le log et alimente une couche softmax. Il est entraîné par rétropropagation à travers le temps (BPTT).

**Pourquoi il est pertinent ici.** Il capture l'**ordre** des tokens (`failed password` ≠ `password failed`), ce que TF-IDF ignore. Les logs étant courts (≈ 10-30 tokens), le problème classique des longues dépendances est atténué, d'où l'intérêt de tester le RNN « simple ».

**Avantages.** Léger (≈ 0,1-0,2 M de paramètres hors embedding), rapide à entraîner par rapport aux Transformers, modèle séquentiel minimal.

**Limites.** Gradients qui s'évanouissent ou explosent, mémoire courte, entraînement séquentiel (peu parallélisable), sensible à l'initialisation et au bruit (Random Swap de SPICE perturbe l'ordre).

**Complexité.** $O(T\cdot (h^2+h\cdot e))$ par séquence et par couche ; prédiction identique ; mémoire $O(T\cdot h)$ pour la BPTT.

**Intérêt dans cette étude.** Le RNN est **absent de l'article** : il constitue l'extension qui isole l'apport des *portes* du LSTM (comparaison à architecture identique).

In [ ]:
if torch_ready():
    for proto in PROTOCOLS:
        run_recurrent("RNN", "RNN", proto)

In [ ]:
if torch_ready():
    for proto in PROTOCOLS:
        show_results("RNN", proto)
        plot_history("RNN", proto)
        interpret("RNN", proto)

**Analyse (à rédiger)** : le RNN fait-il mieux que NB et moins bien que LR/SVM, ou l'inverse ? Examiner les courbes d'apprentissage (instabilité, sur-apprentissage) et la sensibilité au protocole `grouped`. Le Random Swap de SPICE dégrade l'ordre : un modèle séquentiel y est plus exposé qu'un modèle de sac de mots.

**Étape suivante** : LSTM, même architecture avec des portes.

### 7.3 LSTM

**Principe théorique.** Le LSTM ajoute à l'état caché une **cellule mémoire** $c_t$ et trois portes (oubli $f_t$, entrée $i_t$, sortie $o_t$) :
$f_t=\sigma(W_f[h_{t-1},x_t])$, $i_t=\sigma(W_i[\cdot])$, $\tilde c_t=\tanh(W_c[\cdot])$, $c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t$, $h_t=o_t\odot\tanh(c_t)$. Le chemin additif de $c_t$ stabilise le gradient.

**Pourquoi il est pertinent ici.** Les auteurs l'utilisent (embedding 50, 2 couches de 128 unités) et le jugent bien adapté aux logs où l'ordre et les relations entre mots ont un sens. Il atteint 0,9715 sur SIEVE_00 et 0,9524 sur logs réels.

**Avantages.** Meilleure mémoire que le RNN, capture des dépendances d'ordre, bon compromis sans pré-entraînement.

**Limites.** Environ **4× plus de paramètres** de couche que le RNN, entraînement séquentiel lent (5 h 10 min dans l'article, sans doute en grande partie à cause de l'implémentation et du matériel), **instable** selon l'entropie dans l'article (chute à 0,9042 pour $p=0{,}3$ sur logs réels).

**Complexité.** $O(4\,T\,(h^2+h\cdot e))$ par couche ; prédiction identique.

**Intérêt par rapport aux autres modèles.** Seul modèle séquentiel *non pré-entraîné* de l'article ; il sert à tester si l'ordre apporte plus que le simple vocabulaire (LR/SVM) sans le coût d'un Transformer.

In [ ]:
if torch_ready():
    for proto in PROTOCOLS:
        run_recurrent("LSTM", "LSTM", proto)

In [ ]:
if torch_ready():
    for proto in PROTOCOLS:
        show_results("LSTM", proto)
        plot_history("LSTM", proto)
        interpret("LSTM", proto)

**Analyse (à rédiger)** : comparer RNN et LSTM **à architecture identique** (l'écart mesure l'effet des portes). Comparer au LSTM de l'article (référence affichée) et discuter l'instabilité observée par les auteurs : tes résultats la reproduisent-ils ? Mentionner le coût temporel face à LR/SVM.

**Étape suivante** : modèles Transformer pré-entraînés (BERT Base puis DistilBERT).

### 7.4 Briques pour BERT / DistilBERT
- Entrée = **log brut** (aucun prétraitement, comme l'article), tokenisé en *WordPiece*.
- `max_length` fixé automatiquement au 99ᵉ percentile des longueurs en tokens (plafonné à `TF_MAX_LEN_CAP=128`) : garde la mémoire GPU sous contrôle.
- Fine-tuning complet, `lr=2e-5` (valeur standard et celle citée par les auteurs), *warm-up* linéaire de 6 %, précision mixte, batch 16 × accumulation 2.
- Les poids pré-entraînés sont téléchargés depuis Hugging Face (connexion Internet requise à la première exécution ; ensuite en cache). Un chemin local est aussi accepté dans `CFG.BERT_CHECKPOINT`.

> **Note de cohérence avec l'article** : les auteurs indiquent à la fois un `lr` de 2e-5 (fine-tuning) et un Adam avec `lr=0,001` ; ce dernier est inadapté au fine-tuning de BERT (divergence probable). On retient 2e-5 et on le documente comme choix de reproduction.

In [ ]:
if HAS_TORCH and HAS_TRANSFORMERS:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    class RawTextDataset(Dataset):
        def __init__(self, texts, labels):
            self.texts, self.labels = list(texts), list(labels)

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, i):
            return self.texts[i], int(self.labels[i])

    def make_tf_collate(tok, max_len):
        def _collate(batch):
            texts, labels = zip(*batch)
            enc = tok(list(texts), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
            enc["labels"] = torch.tensor(labels, dtype=torch.long)
            return enc
        return _collate

    def tf_forward(model, batch, device):
        return model(input_ids=batch["input_ids"].to(device),
                     attention_mask=batch["attention_mask"].to(device)).logits

    def run_transformer(name: str, checkpoint: str, protocol: str):
        require_eda()
        set_seed()
        tr, te = SPLITS[protocol]
        tr = strat_subsample(tr, CFG.TF_MAX_TRAIN)
        tr_fit, tr_val = split_val(tr)
        norm = lambda s: re.sub(r"\s+", " ", s).strip()                     # seul traitement : espaces
        tok = AutoTokenizer.from_pretrained(checkpoint)
        sample = tr_fit["text"].sample(min(2000, len(tr_fit)), random_state=CFG.SEED).map(norm).tolist()
        tok_lens = [len(x) for x in tok(sample, add_special_tokens=True)["input_ids"]]
        max_len = int(np.clip(np.percentile(tok_lens, 99), 32, CFG.TF_MAX_LEN_CAP))
        collate = make_tf_collate(tok, max_len)

        def make_loader(df, shuffle):
            ds = RawTextDataset(df["text"].map(norm), df["y"].to_numpy())
            return DataLoader(ds, batch_size=CFG.TF_BATCH, shuffle=shuffle, collate_fn=collate)

        train_loader, val_loader, test_loader = make_loader(tr_fit, True), make_loader(tr_val, False), make_loader(te, False)
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=N_CLASSES)
        use_amp = DEVICE == "cuda"
        print(f"[{name}|{protocol}] checkpoint={checkpoint} | params={count_params(model):.1f} M | "
              f"max_len={max_len} (p99 tokens) | train={len(tr_fit)} | amp={use_amp} | device={DEVICE}")
        t0 = time.perf_counter()
        hist = fit_torch(model, train_loader, val_loader, tf_forward, epochs=CFG.TF_EPOCHS, lr=CFG.TF_LR,
                         device=DEVICE, weight_decay=0.01, grad_accum=CFG.TF_ACCUM, use_amp=use_amp,
                         warmup=True, patience=2, desc=name)
        t_train = time.perf_counter() - t0
        t0 = time.perf_counter()
        y_true, y_pred = predict_torch(model, test_loader, tf_forward, DEVICE, use_amp)
        t_infer = time.perf_counter() - t0
        record(name, protocol, y_true, y_pred, t_train, t_infer,
               f"{checkpoint}, max_len={max_len}, train={len(tr_fit)}, epochs={len(hist)}, params={count_params(model):.0f}M")
        HISTORIES[(name, protocol)] = hist
        print(f"[{name:<10}|{protocol:<12}] F1 macro={RESULTS[-1]['f1_macro']:.4f} | acc={RESULTS[-1]['accuracy']:.4f} "
              f"| train={t_train:.0f}s | infer={t_infer:.1f}s")
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

### 7.5 BERT Base

**Principe théorique.** BERT est un encodeur Transformer bidirectionnel (12 couches, 12 têtes, 768 unités cachées, ≈ 110 M de paramètres) pré-entraîné par *Masked Language Modeling* (≈ 15 % des tokens masqués) puis ajusté (*fine-tuning*) avec une tête de classification softmax. L'auto-attention $\mathrm{softmax}(QK^\top/\sqrt{d})V$ relie chaque token à tous les autres.

**Pourquoi il est pertinent ici.** Il prend en compte le contexte dans les deux sens et part de connaissances linguistiques pré-apprises. Dans l'article, BERT est **le meilleur sur logs synthétiques** (0,9528-0,9730) et le plus rapide parmi les modèles profonds, mais **s'effondre sur logs réels** (0,8864-0,9182).

**Avantages.** Représentations contextuelles riches, très bon sur le jeu synthétique, converge en 1-2 époques.

**Limites.** ≈ 110 M de paramètres → risque de **sur-apprentissage sur des gabarits synthétiques** (les auteurs l'évoquent), vocabulaire WordPiece mal adapté aux jetons techniques (adresses, chemins), coût mémoire (4 Go de VRAM : précision mixte, `max_len` court, batch réduit), inférence plus coûteuse.

**Complexité.** Par couche : $O(L^2 d + L d^2)$ avec $L$ la longueur de séquence et $d=768$ ; au total $O(12\,(L^2 d+L d^2))$. Mémoire d'entraînement ≈ paramètres + gradients + états Adam ≈ 1,8 Go en fp32 avant activations.

**Intérêt par rapport aux autres modèles.** C'est le point de comparaison « état de l'art » ; la question de recherche est de savoir si son avantage sur SIEVE est **réel** ou dû à la fuite de gabarit (comparer `paper_random`, `grouped`, `real_test`).

> **Durée indicative** (RTX 3050 Ti, 30 000 logs d'entraînement, 2 époques) : de l'ordre de 10 à 20 min par protocole. Réduis `CFG.TF_MAX_TRAIN` ou lance `QUICK_MODE=True` pour un essai. En cas d'erreur CUDA *out of memory* : `CFG.TF_BATCH = 8` et `CFG.TF_ACCUM = 4`.

In [ ]:
if torch_ready(need_transformers=True):
    for proto in PROTOCOLS:
        run_transformer("BERT", CFG.BERT_CHECKPOINT, proto)

In [ ]:
if torch_ready(need_transformers=True):
    for proto in PROTOCOLS:
        if ("BERT", proto) in PREDS:
            show_results("BERT", proto)
            plot_history("BERT", proto)
            interpret("BERT", proto)

**Analyse (à rédiger)** : BERT surpasse-t-il LR/SVM sur `paper_random` (comme dans l'article) ? Que devient cet avantage sur `grouped` et sur `real_test` ? Comparer l'écart au cas de l'article (0,9730 → 0,9182). Si l'effondrement se reproduit, c'est un argument chiffré pour la thèse « surapprentissage aux artefacts de génération ». Rapporter l'intervalle de confiance sur `real_test`.

**Étape suivante** : DistilBERT, version allégée.

### 7.6 DistilBERT

**Principe théorique.** DistilBERT est une version **distillée** de BERT (Sanh et al., 2019) : 6 couches au lieu de 12 (≈ 66 M de paramètres), entraînée par distillation de connaissances (perte combinant MLM, divergence KL avec les sorties du « professeur » BERT et similarité cosinus des états cachés). D'après ses auteurs, il conserve environ 97 % des performances de BERT sur GLUE, pour un modèle ≈ 40 % plus petit et ≈ 60 % plus rapide.

**Pourquoi il est pertinent ici.** Dans un SIEM, la latence et le coût d'inférence comptent (flux de millions de logs). Moins de paramètres peut aussi réduire le sur-apprentissage aux gabarits synthétiques observé pour BERT.

**Avantages.** Plus rapide à entraîner et à servir, plus léger en mémoire (confortable sur 4 Go), performances proches de BERT.

**Limites.** Capacité réduite (moins de couches), pas de `token_type_ids`, mêmes limites de tokenisation WordPiece que BERT.

**Complexité.** $\approx O(6\,(L^2 d+L d^2))$ : environ la moitié du coût par exemple de BERT Base.

**Intérêt par rapport aux autres modèles.** Absent de l'article : il teste si un Transformer plus petit **généralise mieux** sur logs réels, ce qui répondrait directement à la limite signalée par les auteurs.

In [ ]:
if torch_ready(need_transformers=True):
    for proto in PROTOCOLS:
        run_transformer("DistilBERT", CFG.DISTIL_CHECKPOINT, proto)

In [ ]:
if torch_ready(need_transformers=True):
    for proto in PROTOCOLS:
        if ("DistilBERT", proto) in PREDS:
            show_results("DistilBERT", proto)
            plot_history("DistilBERT", proto)
            interpret("DistilBERT", proto)

**Analyse (à rédiger)** : DistilBERT garde-t-il les performances de BERT (≈ 97 %) pour environ la moitié du temps ? Généralise-t-il mieux sur `real_test` ? Résumer dans une phrase le compromis performance / robustesse / coût.

**Étape suivante** : comparaison globale de tous les modèles.

## 8. Comparaison globale des modèles

Tableau récapitulatif, comparaison des **F1 macro** et des **temps d'entraînement** par modèle et par protocole. Le temps est en échelle logarithmique car les écarts vont de la seconde (NB, SGD) à plusieurs minutes (SVM-RBF, LSTM, BERT).

In [ ]:
res = results_df()
assert len(res), "Aucun résultat : exécute d'abord les sections modèles."
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "train_time_s", "infer_time_s"]

summary = (res.drop_duplicates(subset=["model", "protocol"], keep="last")
              .set_index(["protocol", "model"])[metric_cols + ["notes"]].round(4))
display(summary)
summary.to_csv(OUT / "results_summary.csv")

order = [m for m in ["Dummy", "NaiveBayes", "LogisticRegression", "LinearSVC", "SVM-RBF", "RNN", "LSTM", "BERT", "DistilBERT"]
         if m in set(res["model"])]
last = res.drop_duplicates(subset=["model", "protocol"], keep="last")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=last, x="model", y="f1_macro", hue="protocol", order=order, ax=axes[0])
axes[0].set(title="Comparaison des F1 macro", xlabel="", ylabel="F1 macro", ylim=(0, 1.02))
axes[0].tick_params(axis="x", rotation=40)
axes[0].legend(loc="lower right", fontsize=8)
sns.barplot(data=last, x="model", y="train_time_s", hue="protocol", order=order, ax=axes[1])
axes[1].set(title="Temps d'entraînement (échelle log)", xlabel="", ylabel="secondes", yscale="log")
axes[1].tick_params(axis="x", rotation=40)
plt.tight_layout()
fig.savefig(OUT / "figures" / "comparison_f1_time.png", dpi=150)
plt.show()

pivot = last.pivot(index="model", columns="protocol", values="f1_macro").reindex(order)
fig, ax = plt.subplots(figsize=(1.8 * pivot.shape[1] + 3, 0.5 * len(pivot) + 2))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", vmin=max(0, np.nanmin(pivot.values) - 0.05), vmax=1, ax=ax)
ax.set(title="F1 macro : modèle × protocole", ylabel="", xlabel="")
plt.tight_layout()
fig.savefig(OUT / "figures" / "comparison_heatmap.png", dpi=150)
plt.show()

if "real_test" in set(last["protocol"]):
    rt = last[last["protocol"] == "real_test"][["model", "f1_macro", "f1_macro_ci95", "n_test"]].set_index("model")
    print("\nTest sur logs RÉELS — F1 macro et IC 95 % bootstrap :")
    display(rt.reindex([m for m in order if m in rt.index]))

**Analyse comparative (à rédiger, comme la section « Results and discussion » de l'article)** :
1. **Classement** des modèles par protocole. Le classement change-t-il entre `paper_random`, `grouped` et `real_test` ? Un changement de rang est le résultat scientifique central.
2. **Compromis performance / temps** : repérer les modèles sur la frontière de Pareto (typiquement LR/LinearSVC/SGD très rapides et robustes ; BERT/DistilBERT rapides sur GPU mais moins robustes ; LSTM lent).
3. **Robustesse** : écart `paper_random` → `real_test` par modèle (comparer à l'article : SVM −0,01 ; BERT −0,05).
4. **Significativité** : sur `real_test` (~440 logs), n'affirmer une supériorité que si les intervalles de confiance ne se chevauchent pas.

## 9. Extensions de recherche

Ces trois expériences prolongent l'article et alimentent la rédaction du mémoire. Elles utilisent des modèles rapides (LR, LinearSVC) pour rester exécutables.

### 9.1 Entropie SPICE : effet du paramètre $p$ sur la généralisation réelle
Reproduction de la logique du Tableau 10 de l'article : on entraîne sur chaque variante SIEVE (`SIEVE_00` … `SIEVE_05`, c'est-à-dire plusieurs valeurs de $p$) et on teste sur le **dataset réel**. L'article observe une baisse modeste du MaF1 avec $p$ pour LR/SVM. Cette expérience nécessite le dataset réel **et** au moins un dataset synthétique ; elle est plus informative avec plusieurs variantes dans `data/`.

In [ ]:
def entropy_sweep() -> pd.DataFrame:
    require_eda()
    if REAL is None:
        print("[skip] Pas de dataset réel : expérience impossible.")
        return pd.DataFrame()
    rows = []
    for name, raw in DATASETS.items():
        if name == REAL_NAME:
            continue
        prep = prepare_frame(clean_frame(raw))
        prep = strat_subsample(prep, CFG.ENTROPY_MAX_TRAIN)
        for mname, mk in {"LogisticRegression": lambda: LogisticRegression(max_iter=300, random_state=CFG.SEED),
                          "LinearSVC": lambda: LinearSVC(max_iter=5000, random_state=CFG.SEED)}.items():
            pipe = Pipeline([("tfidf", make_tfidf()), ("clf", mk())]).fit(prep["clean"], prep["y"])
            pred = pipe.predict(REAL["clean"])
            rows.append({"train_dataset": name, "model": mname, "n_train": len(prep),
                         "accuracy_real": accuracy_score(REAL["y"], pred),
                         "f1_macro_real": f1_score(REAL["y"], pred, average="macro", zero_division=0)})
            print(f"  {name:<14}|{mname:<18} accuracy réel={rows[-1]['accuracy_real']:.4f}")
    return pd.DataFrame(rows)


entropy_df = entropy_sweep()
if len(entropy_df):
    display(entropy_df.pivot(index="model", columns="train_dataset", values="accuracy_real").round(4))
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.lineplot(data=entropy_df, x="train_dataset", y="accuracy_real", hue="model", marker="o", ax=ax)
    ax.set(title="Accuracy sur logs réels selon la variante d'entraînement (entropie croissante)", xlabel="", ylabel="accuracy (= micro-F1)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    fig.savefig(OUT / "figures" / "ext_entropy_sweep.png", dpi=150)
    plt.show()
    entropy_df.to_csv(OUT / "ext_entropy_sweep.csv", index=False)

**Analyse (à rédiger)** : la performance sur logs réels décroît-elle avec $p$ comme dans l'article, ou une certaine entropie *améliore-t-elle* la généralisation (compromis fidélité/diversité décrit par les auteurs) ? Avec un seul dataset synthétique, cette analyse se limite à un point.

### 9.2 Fuite de gabarit : quantification de l'optimisme du protocole de l'article
On compare, pour chaque modèle, le F1 macro sous `paper_random` et sous `grouped`. L'écart est une **estimation de l'optimisme** introduit par la présence de variantes d'un même gabarit en train et en test.

In [ ]:
res = results_df()
if {"paper_random", "grouped"} <= set(res["protocol"]):
    p = (res.drop_duplicates(subset=["model", "protocol"], keep="last")
            .pivot(index="model", columns="protocol", values="f1_macro"))
    p = p.drop(index="Dummy", errors="ignore")
    p["optimisme (random - grouped)"] = p["paper_random"] - p["grouped"]
    p = p.sort_values("optimisme (random - grouped)", ascending=False)
    display(p.round(4))
    fig, ax = plt.subplots(figsize=(9, 4))
    p["optimisme (random - grouped)"].plot.bar(ax=ax, color="#c0504d")
    ax.set(title="Optimisme du split aléatoire (F1 macro random − grouped)", xlabel="", ylabel="écart de F1 macro")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    fig.savefig(OUT / "figures" / "ext_leakage_gap.png", dpi=150)
    plt.show()
    p.to_csv(OUT / "ext_leakage_gap.csv")
else:
    print("[skip] Il faut des résultats pour les protocoles 'paper_random' et 'grouped'.")

**Analyse (à rédiger)** : quels modèles perdent le plus en passant à `grouped` ? Un modèle à grande capacité (Transformer, SVM-RBF) qui mémorise les gabarits devrait perdre davantage qu'un modèle linéaire. Rappeler la limite : la clé de groupe est une **approximation sensible à son seuil** ; l'écart mesuré est un ordre de grandeur, pas une mesure exacte de la fuite de gabarit (il peut aussi refléter le retrait de sous-familles entières du train).

### 9.3 Robustesse au bruit : perturbations de type EDA sur le jeu de test
SPICE utilise Synonym Replacement, Random Insertion et Random Swap, mais **exclut** Random Deletion (risque de perte d'information). On perturbe ici le **test** avec *Random Swap* et *Random Deletion* à intensité $p\in\{0;0,1;0,2;0,3;0,5\}$ et on mesure la dégradation des modèles classiques déjà entraînés (protocole `grouped`, à défaut le premier disponible). Cela teste directement l'hypothèse des auteurs sur le caractère délicat de la suppression de tokens.

In [ ]:
def perturb(text: str, p: float, mode: str, rng: np.random.Generator) -> str:
    toks = text.split()
    if len(toks) < 2 or p <= 0:
        return text
    n = max(1, int(round(p * len(toks))))
    if mode == "swap":
        for _ in range(n):
            i, j = rng.integers(0, len(toks), 2)
            toks[i], toks[j] = toks[j], toks[i]
    elif mode == "delete":
        keep = [t for t in toks if rng.random() > p]
        toks = keep or toks[:1]
    return " ".join(toks)


robust_proto = "grouped" if "grouped" in PROTOCOLS else PROTOCOLS[0]
rng = np.random.default_rng(CFG.SEED)
te_df = SPLITS[robust_proto][1]
rows = []
for mname in ("LogisticRegression", "LinearSVC", "NaiveBayes"):
    pipe = MODELS.get((mname, robust_proto))
    if pipe is None:
        continue
    for mode in ("swap", "delete"):
        for p in (0.0, 0.1, 0.2, 0.3, 0.5):
            noisy = te_df["text"].map(lambda t: perturb(t, p, mode, rng)).map(preprocess)
            pred = pipe.predict(noisy)
            rows.append({"model": mname, "perturbation": mode, "p": p,
                         "f1_macro": f1_score(te_df["y"], pred, average="macro", zero_division=0)})
robust_df = pd.DataFrame(rows)
if len(robust_df):
    g = sns.relplot(data=robust_df, x="p", y="f1_macro", hue="model", col="perturbation", kind="line",
                    marker="o", height=3.8, aspect=1.3)
    g.figure.suptitle(f"Robustesse au bruit sur le test [{robust_proto}]", y=1.03)
    g.figure.savefig(OUT / "figures" / "ext_noise_robustness.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(robust_df.pivot_table(index=["perturbation", "model"], columns="p", values="f1_macro").round(4))
    robust_df.to_csv(OUT / "ext_noise_robustness.csv", index=False)
else:
    print("[skip] Entraîne d'abord LR / LinearSVC / NB.")

**Analyse (à rédiger)** : le *swap* dégrade-t-il moins que la *deletion* ? Un modèle de sac de mots (LR/SVM) devrait être quasi insensible au swap (l'ordre n'entre pas dans TF-IDF) mais sensible à la deletion si le mot-clé est supprimé. Cela confirme ou nuance le choix des auteurs d'exclure Random Deletion.

### 9.4 Analyse d'erreurs qualitative
Pour le meilleur modèle sur `real_test` (ou `grouped`), on affiche des exemples mal classés. C'est la base du paragraphe « Limites et erreurs » du mémoire : erreur du modèle, bruit d'annotation ou ambiguïté intrinsèque ?

In [ ]:
res = results_df()
proto_err = "real_test" if "real_test" in set(res["protocol"]) else robust_proto
cands = res[(res["protocol"] == proto_err) & (res["model"] != "Dummy")]
best_model = cands.sort_values("f1_macro", ascending=False).iloc[0]["model"]
y_true, y_pred = PREDS[(best_model, proto_err)]
te_eval = SPLITS[proto_err][1].reset_index(drop=True)
pred_names = np.array([le.classes_[i] for i in y_pred])
err = te_eval.assign(predit=pred_names)[te_eval["label"].to_numpy() != pred_names]
print(f"Meilleur modèle sur [{proto_err}] : {best_model} | {len(err)} erreurs sur {len(te_eval)} ({100 * len(err) / len(te_eval):.1f} %)")
display(err[["text", "label", "predit"]].head(15))

## 10. Export, reproductibilité et conclusion

### 10.1 Export des résultats et de la configuration
Tous les résultats (`results_all_runs.csv`, `results_summary.csv`), les figures et la configuration (`config.json`) sont sauvegardés dans `./outputs/` pour être cités dans le mémoire.

In [ ]:
results_df().to_csv(OUT / "results_all_runs.csv", index=False)
cfg_dump = {k: (v if isinstance(v, (int, float, str, bool, type(None), list)) else str(v)) for k, v in vars(CFG).items()}
env_info = {"python": sys.version.split()[0], "platform": platform.platform(), "pandas": pd.__version__,
            "numpy": np.__version__, "device": DEVICE,
            "torch": torch.__version__ if HAS_TORCH else None,
            "active_dataset": ACTIVE_NAME, "real_dataset": REAL_NAME, "protocols": PROTOCOLS,
            "n_classes": N_CLASSES, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
(OUT / "config.json").write_text(json.dumps({"CFG": cfg_dump, "ENV": env_info}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Fichiers exportés dans", OUT.resolve())
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT))

### 10.2 Trame de conclusion (à compléter avec tes chiffres)
1. **Reproduction** : les ordres de grandeur de l'article (SVM/LR/BERT/LSTM ≈ 0,95-0,97 sur SIEVE) sont-ils retrouvés sous `paper_random` ?
2. **Optimisme du protocole** : l'écart `paper_random` → `grouped` chiffre la fuite de gabarit.
3. **Généralisation réelle** : classement sur `real_test` avec intervalles de confiance ; BERT/DistilBERT s'effondrent-ils comme dans l'article ?
4. **Compromis opérationnel** : performance, robustesse, temps d'entraînement et d'inférence (contexte SIEM : latence).
5. **Limites** : clé de groupe approximative ; SVC RBF sur sous-échantillon ; Transformers entraînés sur sous-échantillon (`TF_MAX_TRAIN`) ; jeu réel minuscule ; une seule graine.

### 10.3 Pistes pour prolonger la recherche
- **Multi-graines** (5 graines) + tests statistiques (McNemar, Wilcoxon) pour fiabiliser les comparaisons.
- **Classification hiérarchique ECS** (catégorie puis action) : l'article note que l'action est plus difficile que la catégorie.
- **Ablation des opérateurs SPICE** (SR, RI, RS seuls) pour isoler leur effet sur la généralisation réelle.
- **Généralisation inter-sources** : entraîner sur Loghub, tester sur OSSEC (si la source est identifiable dans `source_file`).
- **Détection de logs inconnus** (open-set / rejet par seuil de confiance), pertinent pour un SIEM confronté à de nouveaux équipements.
- **Données réelles supplémentaires** : élargir le test externe, car 440 logs limitent fortement la puissance statistique.